# C03–M1–L2 — Similarity Search on Text Using Chroma DB and Python (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original files](https://github.com/hengkuankian/learning_ibm_rag_agentic/blob/main/Course3_Vector_Databases_for_RAG/Similarity_Search_on_Text_Using_Chroma_DB_and_Python/similarity_search.py) · cross-checked with the [Hugging Face reference port](https://github.com/oooookk7/huggingface-rag-and-agentic-ai/tree/main/03-vector-databases-for-rag)

**Before you run:** add **`GOOGLE_API_KEY`** to Colab 🔑 Secrets (free key from aistudio.google.com/apikey). CPU runtime is enough — no GPU, no model downloads.

**What changed vs. the IBM version (CP-G.3 Gemini migration)**
- Setup: Cell 1 = minimal pip install (+ `chromadb`), Cell 2 = universal Gemini init. Legacy Groq/HF setup, `get_llm()` and watsonx shim removed; no `sentence-transformers` download.
- New `%%writefile gemini_ef.py`: `GeminiEmbeddingFunction` (Chroma embedding function over `models/gemini-embedding-2`).
- `similarity_search.py` recreated with `%%writefile` as before; only change: `SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")` → `GeminiEmbeddingFunction()`.
- Colab exercises: interactive queries, cosine vs L2 vs IP, `gemini-embedding-001` vs `gemini-embedding-2`, persistent storage, mini RAG with `llm.invoke(...).text`.

Everything else — explanations, code, exercises — is the original lab.

**Review fixes (Oct 13, 2026):** C2 no longer deletes the collection used by C5 (`NotFoundError`), IBM lab *Practice Exercises 1–3* added as a task, Colab exercises C1–C5 now have starter cells + hidden ✅ Answer cells.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai chromadb

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

> 💡 **How to use the ✅ Answer cells:** every exercise is followed by a hidden, collapsed cell titled *✅ Answer — …*. Try the exercise yourself first. To see the reference answer, double-click the cell (or *Show code*), tick **run_answer** and run it. While the box is unticked the cell raises a short *Answer not run* message on purpose, so **Runtime ▸ Run all** stops right before the first answer instead of spoiling it — after that, run the remaining cells one by one (or tick the boxes you want and use *Run after*).

### `gemini_ef.py` — Gemini embedding function for Chroma
A tiny `chromadb.EmbeddingFunction` over `GoogleGenerativeAIEmbeddings` (swap rule: *Raw ChromaDB → GeminiEmbeddingFunction*). The lab scripts import it instead of `SentenceTransformerEmbeddingFunction`; they read `GOOGLE_API_KEY` from the environment set in Cell 2.

In [ ]:
%%writefile gemini_ef.py
# Chroma embedding function backed by Gemini (replaces SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2"))
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
from chromadb import Documents, EmbeddingFunction, Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str = "models/gemini-embedding-2"):
        self.model = model
        self._emb = GoogleGenerativeAIEmbeddings(model=model)   # key read from GOOGLE_API_KEY

    def __call__(self, input: Documents) -> Embeddings:
        texts = list(input)
        out = []
        for i in range(0, len(texts), 100):              # free tier: <= 100 texts per call
            out += self._emb.embed_documents(texts[i:i + 100])
        return out

    @staticmethod
    def name() -> str:
        return "gemini"

    def get_config(self):
        return {"model": self.model}

    @staticmethod
    def build_from_config(config):
        return GeminiEmbeddingFunction(config.get("model", "models/gemini-embedding-2"))

## The lab script
Steps from the lab: create the embedding function and client → create the `my_grocery_collection` collection (cosine HNSW space) → add 14 grocery texts with ids and metadata → `collection.get()` → `perform_similarity_search()` for the query terms `["red", "fresh"]`. Edit `query_term` in the file and rerun to try your own queries (lab exercise).

In [ ]:
%%writefile similarity_search.py
# Importing the necessary modules from the chromadb package:
# chromadb is used to interact with the Chroma DB database,
# embedding_functions is used to define the embedding model
import chromadb
from chromadb.utils import embedding_functions
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings

# Function to perform a similarity search in the collection
def perform_similarity_search(collection, all_items):
    try:
        # Place your similarity search code inside this block
        # Define the query term you want to search for in the collection
        # query_term = "apple"
        query_term=["red", "fresh"]

        # Perform a query to search for the most similar documents to the 'query_term'
        results = collection.query(
            query_texts=query_term,
            n_results=3  # Retrieve top 3 results
        )

        

        print(f"Query results for '{query_term}':")
        print(results)

        # Check if no results are returned or if the results array is empty
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            # Log a message indicating that no similar documents were found for the query term
            print(f'No documents found similar to "{query_term}"')
            return

        for q in range(len(query_term)):
            print(f'Top 3 similar documents to "{query_term[q]}":')
            # Access the nested arrays in 'results["ids"]' and 'results["distances"]'
            for i in range(min(3, len(results['ids'][q]))):
                doc_id = results['ids'][q][i]  # Get ID from 'ids' array
                score = results['distances'][q][i]  # Get score from 'distances' array
                # Retrieve text data from the results
                text = results['documents'][q][i]
                if not text:
                    print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
                else:
                    print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')
    except Exception as error:
        print(f"Error in similarity search: {error}")


# Define the embedding function (Colab: Gemini instead of SentenceTransformers all-MiniLM-L6-v2)
ef = GeminiEmbeddingFunction()

# Create a new instance of ChromaClient to interact with the Chroma DB
client = chromadb.Client()
# Define the name for the collection to be created or retrieved
collection_name = "my_grocery_collection"

# Define the main function to interact with the Chroma DB
def main():
    try:
        # Place your database operations inside this block
        
        # Create a collection in the Chroma database with a specified name, 
        # distance metric, and embedding function. In this case, we are using 
        # cosine distance
        collection = client.create_collection(
            name=collection_name,
            metadata={"description": "A collection for storing grocery data"},
            configuration={
                "hnsw": {"space": "cosine"},
                "embedding_function": ef
            }
        )
        print(f"Collection created: {collection.name}")
        pass
    except Exception as error:  # Catch any errors and log them to the console
        print(f"Error: {error}")

    # Array of grocery-related text items
    texts = [
        'fresh red apples',
        'organic bananas',
        'ripe mangoes',
        'whole wheat bread',
        'farm-fresh eggs',
        'natural yogurt',
        'frozen vegetables',
        'grass-fed beef',
        'free-range chicken',
        'fresh salmon fillet',
        'aromatic coffee beans',
        'pure honey',
        'golden apple',
        'red fruit'
    ]

    # Create a list of unique IDs for each text item in the 'texts' array
    # Each ID follows the format 'food_<index>', where <index> starts from 1
    ids = [f"food_{index + 1}" for index, _ in enumerate(texts)]

    # Add documents and their corresponding IDs to the collection
    # The `add` method inserts the data into the collection
    # The documents are the actual text items, and the IDs are unique identifiers
    # ChromaDB will automatically generate embeddings using the configured embedding function
    collection.add(
        documents=texts,
        metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts],
        ids=ids
    )

    # Retrieve all the items (documents) stored in the collection
    # The `get` method fetches all data from the collection
    all_items = collection.get()
    # Log the retrieved items to the console for inspection
    # This will print out all the documents, IDs, and metadata stored in the collection
    print("Collection contents:")
    print(f"Number of documents: {len(all_items['documents'])}")

    perform_similarity_search(collection, all_items)

if __name__ == "__main__":
    main()

In [ ]:
!python similarity_search.py

## Lab Practice Exercises (from the IBM lab instructions)
The lab first runs the script with `query_term = "apple"` (Task 6: top 3 documents similar to *apple*), then asks you to search for several queries at once, `query_term = ["red", "fresh"]`. *(The `%%writefile` script above already contains IBM's final version — try to rebuild it yourself from the `"apple"` version.)*

1. Replace `query_term = "apple"` with `query_term = ["red", "fresh"]`. Moreover, write code that **converts `query_term` to a list if it is a string**.
2. Change `[query_term]` to `query_term` in the call to `collection.query()`.
3. Modify the block that retrieves the top documents to allow for multiple search queries. *Hint: use an outer loop around the whole block.*
4. Verify that the output is correct (run the function for `"apple"` and for `["red", "fresh"]`).

In [ ]:
# Your code here
# start from query_term = "apple" and make perform_similarity_search() handle a list of queries

In [ ]:
# @title ✅ Answer — Lab Practice Exercises 1–3 · multiple query terms
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
import chromadb
from gemini_ef import GeminiEmbeddingFunction
texts = ['fresh red apples','organic bananas','ripe mangoes','whole wheat bread','farm-fresh eggs','natural yogurt',
         'frozen vegetables','grass-fed beef','free-range chicken','fresh salmon fillet','aromatic coffee beans',
         'pure honey','golden apple','red fruit']
lab_client = chromadb.Client()
lab_col = lab_client.get_or_create_collection(
    "lab_practice_grocery",
    configuration={"hnsw": {"space": "cosine"}, "embedding_function": GeminiEmbeddingFunction()})
lab_col.upsert(documents=texts, ids=[f"food_{i + 1}" for i in range(len(texts))],
               metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts])

def perform_similarity_search(collection, all_items, query_term=["red", "fresh"]):   # Ex. 1
    try:
        if isinstance(query_term, str):          # Ex. 1: a single string -> list with one query
            query_term = [query_term]
        results = collection.query(
            query_texts=query_term,              # Ex. 2: no extra [ ] around query_term
            n_results=3)
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            print(f'No documents found similar to "{query_term}"')
            return
        for q in range(len(query_term)):         # Ex. 3: outer loop = one block per query
            print(f'Top 3 similar documents to "{query_term[q]}":')
            for i in range(min(3, len(results['ids'][q]))):
                doc_id = results['ids'][q][i]
                score = results['distances'][q][i]
                text = results['documents'][q][i]
                if not text:
                    print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
                else:
                    print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')
    except Exception as error:
        print(f"Error in similarity search: {error}")

all_items = lab_col.get()
perform_similarity_search(lab_col, all_items, "apple")            # Ex. 4: string still works
perform_similarity_search(lab_col, all_items, ["red", "fresh"])   # Ex. 4: two queries

## 🧪 Colab exercises
### C1 · Query interactively in the notebook
Import the script's objects and keep the collection in memory so you can query without re-embedding.

In [ ]:
# C1 setup (given): in-notebook client + helper that (re)builds a grocery collection
import chromadb
from chromadb.utils import embedding_functions
from gemini_ef import GeminiEmbeddingFunction  # Colab: Gemini embeddings
ef = GeminiEmbeddingFunction()
client = chromadb.Client()
texts = ['fresh red apples','organic bananas','ripe mangoes','whole wheat bread','farm-fresh eggs','natural yogurt',
         'frozen vegetables','grass-fed beef','free-range chicken','fresh salmon fillet','aromatic coffee beans',
         'pure honey','golden apple','red fruit']
def build(name, space="cosine", fn=ef):
    try: client.delete_collection(name)
    except Exception: pass
    col = client.create_collection(name=name, configuration={"hnsw": {"space": space}, "embedding_function": fn})
    col.add(documents=texts, ids=[f"food_{i+1}" for i in range(len(texts))],
            metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts])
    return col
col = build("grocery_cosine")   # used again in C5
print(col.name, col.count(), "docs")

**Your task:** query `col` with 2–3 phrases of your own (e.g. *something sweet for breakfast*, *seafood*, *a hot drink*) and print each top-3 document with its distance.

In [ ]:
# Your code here
# query `col` with your own phrases

In [ ]:
# @title ✅ Answer — C1 · query interactively
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
for q in ["something sweet for breakfast", "seafood", "a hot drink"]:
    r = col.query(query_texts=[q], n_results=3)
    print(f"{q!r:32} →", list(zip(r["documents"][0], [round(d, 3) for d in r["distances"][0]])))

### C2 · Cosine vs L2 vs inner product
Build the same collection with `space="l2"` and `space="ip"`. Do the top-3 results change? Do the distance values?

In [ ]:
# Your code here
# build the collection with space="l2" and space="ip" (use new collection names!) and compare

In [ ]:
# @title ✅ Answer — C2 · cosine vs L2 vs inner product
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
res = {}
for space in ["cosine", "l2", "ip"]:
    r = build(f"cmp_{space}", space).query(query_texts=["red"], n_results=3)   # own names: keeps C1's `col` alive
    res[space] = r
    print(f"{space:6}", list(zip(r["documents"][0], [round(d, 3) for d in r["distances"][0]])))
same = res["cosine"]["documents"][0] == res["l2"]["documents"][0] == res["ip"]["documents"][0]
print("same top-3 order:", same)
cos_d = res["cosine"]["distances"][0]
print("ip distance == cosine distance:", all(abs(a - b) < 1e-4 for a, b in zip(res["ip"]["distances"][0], cos_d)))
print("l2 distance == 2 × cosine distance:", all(abs(a - 2 * b) < 1e-4 for a, b in zip(res["l2"]["distances"][0], cos_d)))

### C3 · A second Gemini embedding model
Swap the embedding function to `models/gemini-embedding-001` and compare with `gemini-embedding-2`.

In [ ]:
# Your code here
# GeminiEmbeddingFunction("models/gemini-embedding-001") + build(...)

In [ ]:
# @title ✅ Answer — C3 · gemini-embedding-001 vs gemini-embedding-2
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
ef_001 = GeminiEmbeddingFunction("models/gemini-embedding-001")
col_001 = build("grocery_gemini_001", fn=ef_001)
queries = ["red", "fresh"]
r1 = col_001.query(query_texts=queries, n_results=3)
r2 = col.query(query_texts=queries, n_results=3)          # C1 collection = gemini-embedding-2
for q, d1, d2 in zip(queries, r1["documents"], r2["documents"]):
    print(f"{q:6} 001 → {d1}\n{'':6} 2   → {d2}   overlap {len(set(d1) & set(d2))}/3")

### C4 · Persist the database
`chromadb.PersistentClient(path=...)` keeps the vectors on disk (mount Google Drive to keep them across sessions).

In [ ]:
# Your code here
# chromadb.PersistentClient(path="chroma_store") ...

In [ ]:
# @title ✅ Answer — C4 · persist the database
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
pclient = chromadb.PersistentClient(path="chroma_store")
pcol = pclient.get_or_create_collection("grocery", configuration={"hnsw": {"space": "cosine"}, "embedding_function": ef})
pcol.upsert(documents=texts, ids=[f"food_{i+1}" for i in range(len(texts))])   # upsert = safe to re-run
print(pcol.count(), "docs stored in ./chroma_store")

# simulate a new session: a fresh client on the same folder sees the data without re-adding it
reopened = chromadb.PersistentClient(path="chroma_store").get_collection("grocery", embedding_function=ef)
print("reopened:", reopened.count(), "docs →", reopened.query(query_texts=["apple"], n_results=1)["documents"][0])

### C5 · Mini RAG
Retrieve the top 3 items and let `llm` answer from them only.

In [ ]:
# Your code here
# retrieve top-3 from `col`, then llm.invoke(...).text

In [ ]:
# @title ✅ Answer — C5 · mini RAG
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸️ Answer not run — try the exercise first, then tick run_answer and run this cell."]
    raise AnswerNotRun("Answer not run")
question = "I want a healthy protein for dinner — what do you have?"
hits = col.query(query_texts=[question], n_results=3)["documents"][0]
print("retrieved:", hits)
prompt = (f"You are a grocery assistant. Answer ONLY from these grocery items: {hits}. "
          f"If none of them fits, say so.\nQuestion: {question}")
print(llm.invoke(prompt).text)        # .text = plain string (Gemini 3 returns a list of content blocks)